<a href="https://colab.research.google.com/github/BilalAsifB/MissenseImpact-FYP/blob/main/notebooks/data-preprocessing/Bilal/process_data_sg10k_session_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import subprocess

INPUT_DIR = "/content/drive/MyDrive/FYP_DATA/DATA/raw_data/SG10K/"
ALIAS_FILE = "/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/indian_alias.txt"
OUTPUT_DIR = "/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered/"


print("Installing bcftools...")
!sudo apt-get update -q
!sudo apt-get install bcftools -y -q

# --- CONFIGURATION ---
chr_num = "21"
# Adjust filename pattern to match yours (e.g., "chr1.vcf.gz" or "sg10k_chr1.vcf.gz")
input_vcf = os.path.join(INPUT_DIR, f"chr{chr_num}.vcf.gz")
output_vcf = os.path.join(OUTPUT_DIR, f"chr{chr_num}_sas_filtered.vcf.gz")

print(f"Processing Chromosome {chr_num}...")
print(f"Input: {input_vcf}")
print(f"Output: {output_vcf}")

# --- VERIFICATION: BEFORE RUNNING ---
print("📊 Pre-processing verification:")
print(f"Total samples in original: {os.popen(f'bcftools query -l {input_vcf} | wc -l').read().strip()}")
print(f"Indian samples requested: {os.popen(f'wc -l < {ALIAS_FILE}').read().strip()}")
print()

# --- THE COMMAND ---
# 1. -S: Keep only Indian samples
# 2. --force-samples: Ignore if ID missing (safety)
# 3. +fill-tags: Recalculate AF, AC, AN for the new subset

cmd = f"""
bcftools view -S "{ALIAS_FILE}" --force-samples -Ou "{input_vcf}" | \
bcftools plugin fill-tags -Oz -o "{output_vcf}" -- -t AF,AC,AN
"""

# Run it
print("⏳ Running bcftools...")
exit_code = os.system(cmd)
print()

if exit_code == 0:
    print(f"✅ Success! Created: {output_vcf}")

    # Index the new file immediately
    print("Indexing...")
    os.system(f"bcftools index -t '{output_vcf}'")

    # Check size
    size_mb = os.path.getsize(output_vcf) / (1024 * 1024)
    print(f"File Size: {size_mb:.2f} MB")

    # --- VERIFICATION: AFTER RUNNING ---
    print()
    print("📊 Post-processing verification:")
    n_samples = os.popen(f"bcftools query -l '{output_vcf}' | wc -l").read().strip()
    print(f"Final sample count: {n_samples}")

else:
    print("❌ Error processing file. Check paths and logs.")

Installing bcftools...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
bcftools is already the newest version (1.13-1).
0 upgraded, 0 newly installed, 0 to remove